# End-to-end analysis from raw reads (Cell Ranger)

**Status: work in progress.**

The breast-cancer analysis in `01_breast_ddr.ipynb` begins from the CZ CELLxGENE atlas, which is
supplied already processed by the original authors: raw counts are present, but read alignment and
quantification were performed upstream. This notebook documents and demonstrates the front of the
pipeline that the processed atlas does not exercise, namely the conversion of raw 10x Genomics
sequencing reads into a cell-by-gene count matrix with Cell Ranger, followed by the same downstream
analysis. It establishes a fully reproducible path from FASTQ files to annotated cells.

## Overview

The end-to-end workflow has five stages:

1. Obtain raw 10x Genomics reads (FASTQ files) for a chosen sample.
2. Obtain a reference transcriptome.
3. Run `cellranger count` to align reads and quantify expression, producing a filtered
   cell-by-gene matrix.
4. Load the resulting matrix into Scanpy.
5. Run the same downstream pipeline as `01_breast_ddr.ipynb` (quality control, normalisation,
   feature selection, dimensionality reduction, clustering, and annotation).

Stages 1 to 3 require raw sequencing data, a reference, and substantial compute (Cell Ranger is run
from the shell, not from Python), and are therefore documented here as commands rather than executed
inline. Stages 4 and 5 run in this environment once a count matrix is available.

## Stage 1: obtain raw reads (FASTQ)

Raw 10x reads are retrieved from a public repository (for example SRA or a 10x Genomics public
dataset). The files must follow the Cell Ranger naming convention
`SAMPLE_S1_L001_R1_001.fastq.gz` and `SAMPLE_S1_L001_R2_001.fastq.gz`.

```bash
# Example: download FASTQ files into fastq/ (accession to be selected)
mkdir -p fastq
# prefetch <SRR_ACCESSION> && fasterq-dump --split-files <SRR_ACCESSION> -O fastq/
```

## Stage 2: reference transcriptome

Cell Ranger requires a prebuilt reference. The 10x Genomics human reference (GRCh38) is used here.

```bash
# Download and extract the 10x human reference (GRCh38)
curl -O "https://cf.10xgenomics.com/supp/cell-exp/refdata-gex-GRCh38-2020-A.tar.gz"
tar -xzf refdata-gex-GRCh38-2020-A.tar.gz
```

## Stage 3: run Cell Ranger

`cellranger count` aligns the reads, calls cells, and quantifies expression. The filtered output is
written to `runs/SAMPLE/outs/filtered_feature_bc_matrix/`.

```bash
cellranger count \
  --id=SAMPLE \
  --fastqs=fastq/ \
  --sample=SAMPLE \
  --transcriptome=refdata-gex-GRCh38-2020-A \
  --output-dir=runs/SAMPLE
```

## Stage 4: load the count matrix into Scanpy

The filtered matrix produced by Cell Ranger is read directly with `scanpy.read_10x_mtx`. This is the
point at which the end-to-end workflow rejoins the downstream pipeline. The cell below runs once a
Cell Ranger output directory is present.

In [ ]:
import scanpy as sc

# Path to the Cell Ranger filtered matrix (set once Stage 3 has been run)
MATRIX_DIR = "runs/SAMPLE/outs/filtered_feature_bc_matrix"

# adata = sc.read_10x_mtx(MATRIX_DIR, var_names="gene_symbols", cache=True)
# adata.var_names_make_unique()
# print(adata)

## Stage 5: downstream analysis

From this point the analysis is identical to `01_breast_ddr.ipynb`: per-cell quality-control metrics
and distribution-based filtering, normalisation (counts per 10,000 and log1p), selection of highly
variable genes, scaling, PCA, neighbour-graph construction, Leiden clustering, UMAP, and
marker-based annotation. The corresponding sections of `01_breast_ddr.ipynb` are reused so that the
only difference between the two notebooks is the source of the count matrix.

## Requirements and next steps

- Cell Ranger (10x Genomics) installed and on the shell path.
- A selected public 10x sample with available FASTQ files.
- The GRCh38 reference and sufficient disk and memory for alignment.

Next step: select a public breast-cancer 10x sample, run Stages 1 to 3, and connect the resulting
matrix to the downstream pipeline.